# Lakeflow Connect: Demostración del Conector de Google Drive

Este notebook demuestra cómo usar el **Conector Estándar de Google Drive de Databricks** (actualmente en Beta) para ingerir varios tipos de archivos desde Google Drive a tablas Delta con gobernanza de Unity Catalog. Haz una copia de este notebook para ejecutarlo tú mismo.

Consulta nuestra documentación pública [aquí](https://docs.google.com/document/d/e/2PACX-1vSV8pfG-Ib0A0QCsDJAUx3EoTlefMc6CVUdFrudgeVU5AcRPmvQggij2gbATuLN58F3b6yIvPS11D2M/pub). 

## Lo Que Aprenderás

* **Ingerir PDFs no estructurados** - Leer archivos PDF y analizarlos usando funciones de IA
* **Sincronizar archivos Excel** - Cargar archivos Excel individuales en tablas Delta
* **Ingerir archivos CSV** - Combinar múltiples archivos estructurados con el mismo esquema
* **Ingesta Incremental Automática usando Lakeflow SDP** - Orquestar toda tu ingesta automáticamente usando Lakeflow Spark Declarative Pipelines

## Requisitos Previos

* **Conexión de Unity Catalog**: Los clientes necesitan crear su propia Conexión UC. Esta demostración interna usa la conexión de Unity Catalog `gdrive_demo_connection`
* **Databricks Runtime**: 17.3 o superior
* Si deseas ingerir Google Sheets y/o archivos Excel, también necesitarás habilitar la característica Beta del formato de archivo Excel. Aprende más [aquí](https://docs.databricks.com/aws/en/query/formats/excel).

## Acerca del Conector de Google Drive

El Conector Estándar de Google Drive soporta:
* Ingesta por lotes y en streaming (Auto Loader, spark.read, COPY INTO)
* Archivos estructurados (ej. CSV, Excel), semi-estructurados (ej. JSON) y no estructurados (ej. PDF, imágenes, documentos)
* Gobernanza y seguridad de Unity Catalog
* Inferencia y evolución automática de esquemas

---

## 1. Ingerir PDFs No Estructurados y Analizarlos con IA

Esta sección demuestra cómo:
1. Leer archivos PDF desde Google Drive como archivos binarios
2. Guardarlos en una tabla Delta
3. Usar funciones de IA SQL para analizar y extraer contenido estructurado de los PDFs

In [0]:
# Read all PDF files from Google Drive as binary files
# This is a batch read. For automatic incremental ingestion, view the cells at the bottom of the notebook to see how to use this connector in Lakeflow Spark Declarative Pipelines
pdf_df = (spark.read
    .format("binaryFile") # Use this format for unstructured data
    .option("databricks.connection", "gdrive_demo_connection") # User provides the name of their connection
    .option("recursiveFileLookup", True)
    .option("pathGlobFilter", "*.pdf")  # Only ingest PDF files
    .load("https://drive.google.com/drive/u/5/folders/11gJMa1Nywn4gkiYrH4CS-aPZjvLBlDHj")
    .select("*", "_metadata")
)

# Save the PDF files to a Delta table for persistent storage
pdf_df.write \
    .mode("overwrite") \
    .saveAsTable("aircraft_maintence_logs")

# Display the DataFrame to see the PDF files
display(pdf_df)

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-3064659049279705>, line 15
      3 pdf_df = (spark.read
      4     .format("binaryFile") # Use this format for unstructured data
      5     .option("databricks.connection", "gdrive_demo_connection") # User provides the name of their connection
   (...)
      9     .select("*", "_metadata")
     10 )
     12 # Save the PDF files to a Delta table for persistent storage
     13 pdf_df.write \
     14     .mode("overwrite") \
---> 15     .saveAsTable("aircraft_maintence_logs")
     17 # Display the DataFrame to see the PDF files
     18 display(pdf_df)

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/readwriter.py:737, in DataFrameWriter.saveAsTable(self, name, format, mode, partitionBy, **options)
    735 self._write.table_name = name
    736 self._write.table_save_method = "save_as_table"
--> 737 _

### Analizar PDFs usando `ai_parse_document()`

Al ingerir archivos no estructurados desde Google Drive (como PDFs, Google Docs o Google Slides) usando el conector estándar de Google Drive con formato binaryFile, el contenido de los archivos se almacena como datos binarios sin procesar.

Para preparar estos archivos para cargas de trabajo de IA—como RAG, búsqueda, clasificación o comprensión de documentos—puedes analizar fácilmente el contenido binario en una salida estructurada y consultable simplemente aplicando `ai_parse_document()` a la columna `content`.

In [0]:
%sql
-- Parse PDF content using AI and extract structured information. 
SELECT
  *,
  ai_parse_document(content) AS parsed_content
FROM
  aircraft_maintence_logs
LIMIT 5 -- limiting to only the first 5 PDFs

path modificationTime length content _metadata parsed_content https://drive.google.com/file/d/10GgH5FPzoeBMv9Kw3ouoGP-VQxYx0aUA 2025-06-08T03:37:14.000Z 153855 ... List(https://drive.google.com/file/d/10GgH5FPzoeBMv9Kw3ouoGP-VQxYx0aUA, logbook_N320CX_60.pdf, 153855, 0, 153855, 2025-06-08T03:37:14.000Z) {"document":{"elements":[{"bbox":[{"coord":[912,86,1123,106],"page_id":0}],"content":"AIRCRAFT MAINTENANCE LOGBOOK","description":null,"id":0,"type":"page_header"},{"bbox":[{"coord":[57,225,1987,2529],"page_id":0}],"content":" N-Number: N320XP Technician Time Total Time Description of Work Performed 01/31/2025 52.5 58.5 Performed operational check of rudder cable tension. Found in nominal condition. Examined fuselage skin. Found in secure condition. I certify that this aircraft has been inspected in accordance with the scope and detail of Part 43 Appendix D and have determined it to be in a condition for safe flight. Courtney Stewart 2910783AJP ","description":null,"id":1,"type":"table"}],"pages":[{"id":0,"image_uri":null}]},"error_status":null,"metadata":{"file_metadata":null,"id":"42e8da03-bf08-4ec1-965d-a1182964348d","version":"2.0"}} https://drive.google.com/file/d/10IvLQDEPUf9Qycc71EOAbqvqI6d72Z0d 2025-06-08T03:37:24.000Z 182430 ... List(https://drive.google.com/file/d/10IvLQDEPUf9Qycc71EOAbqvqI6d72Z0d, logbook_N763PZ_86.pdf, 182430, 0, 182430, 2025-06-08T03:37:24.000Z) {"document":{"elements":[{"bbox":[{"coord":[924,86,1130,106],"page_id":0}],"content":"AIRCRAFT MAINTENANCE LOGBOOK","description":null,"id":0,"type":"page_header"},{"bbox":[{"coord":[103,216,517,266],"page_id":0}],"content":"N-Number: N763PU","description":null,"id":1,"type":"text"},{"bbox":[{"coord":[66,266,1992,2567],"page_id":0}],"content":" Date Technician Time Total Time Description of Work Performed 03/16/2025 138.0 152.7 Performed Annual/Condition Inspection IAW Part 43, Appendix D. Verified pilot-side door latch. Found in airworthiness condition. After abnormal engine reading, found carburetor dirty. Overhauled component IAW service manual. Returned to service. After abnormal engine reading, found #3 cylinder CHT probe sluggish. Lubricated component IAW service manual. Ops check good. I certify that this aircraft has been inspected in accordance with the scope and detail of Part 43 Appendix D and have ","description":null,"id":2,"type":"table"}],"pages":[{"id":0,"image_uri":null}]},"error_status":null,"metadata":{"file_metadata":null,"id":"9387b84c-e12d-4509-aeaa-b2e0edd4c935","version":"2.0"}} https://drive.google.com/file/d/12DWDZX-QlU5fO6e8xI0cFWDDbDwvRzWl 2025-06-08T03:37:18.000Z 159268 ... List(https://drive.google.com/file/d/12DWDZX-QlU5fO6e8xI0cFWDDbDwvRzWl, logbook_N473QQ_54.pdf, 159268, 0, 159268, 2025-06-08T03:37:18.000Z) {"document":{"elements":[{"bbox":[{"coord":[913,86,1125,113],"page_id":0}],"content":"AIRCRAFT MAINTENANCE LOGBOOK","description":null,"id":0,"type":"page_header"},{"bbox":[{"coord":[94,216,1987,2537],"page_id":0}],"content":" N-Number: N473QL Technician Time Total Time Description of Work Performed 01/17/2025 473.0 487.7 During routine inspection, found main bus sluggish. Serviced component IXW service manual. Ops check good. Inspected nose wheel strut. Found in within specified tolerance condition. I certify that this aircraft has been inspected in accordance with the service and detail of Part 43 appendix D and have determined it to be in a condition for safe flight. Kevin Lopez 7714678AJP ","description":null,"id":1,"type":"table"}],"pages":[{"id":0,"image_uri":null}]},"error_status":null,"metadata":{"file_metadata":null,"id":"22af3945-757f-4f61-844c-e600d0457a2e","version":"2.0"}} https://drive.google.com/file/d/13Hszvo-cU8106AoCXPP-tZbPxYB_d101 2025-06-08T03:37:20.000Z 154594 ... List(https://drive.google.com/file/d/13Hszvo-cU8106AoCXPP-tZbPxYB_d101, logbook_N632XU_45.pdf, 154594, 0, 154594, 2025-06-08T03:37:20.000Z) {"document":{"elements":[{"bbox":[{"coord":[915,89,1124,113],"page_id":0}],"content":"AIRCRAFT MAINTE

También puedes usar `ai_parse_document()` **dentro de Lakeflow Spark Declarative Pipelines para habilitar el análisis incremental**. A medida que llegan nuevos archivos desde Google Drive, se analizan automáticamente cuando tu pipeline se actualiza.

---

## 2. Sincronizar Google Sheets a Tablas Delta

Esta sección demuestra cómo:
1. Leer una Google Sheet específica desde Google Drive usando Python `spark.read()` o SQL `read_files()`

Las Google Sheets se tratan como archivos Excel. Los archivos Excel soportan opciones como `headerRows` y `dataAddress` para especificar qué hoja y rango leer. Consulta la lista completa de opciones de análisis [aquí](https://docs.databricks.com/aws/en/query/formats/excel).

In [0]:
# Read a specific Google Sheet
gsheet_df = (spark.read
    .format("excel")
    .option("databricks.connection", "gdrive_demo_connection")
    .option("headerRows", 1)  # First row contains headers
    .option("inferSchema", True)  # Automatically infer column types
    .option("dataAddress", "'Sheet1'!A1:D15") # Parse specific cells
    .load("https://docs.google.com/spreadsheets/d/1CQO4AzMthL-w0XWAcSkw4Eox6bo64rWVPCpzUflSsFA/edit?gid=0#gid=0")
)

# Save the Excel data to a Delta table
gsheet_df.write \
    .mode("overwrite") \
    .saveAsTable("gsheet_table")

# Display the Excel data
display(gsheet_df)

ORDERNUMBER,QUANTITYORDERED,PRICEEACH,ORDERLINENUMBER
10107,30,95.70,2
10121,34,81.35,5
10134,41,94.74,2
10145,45,83.26,6
10159,49,100.00,14
10168,36,96.66,1
10180,29,86.13,9
10188,48,100.00,1
10201,22,98.57,2
10211,41,100.00,14


In [0]:
%sql
-- Create a table from a Google Sheet file using SQL
CREATE OR REPLACE TABLE gsheet_table_sql AS
SELECT * FROM read_files(
  "https://docs.google.com/spreadsheets/d/1CQO4AzMthL-w0XWAcSkw4Eox6bo64rWVPCpzUflSsFA/edit?gid=87243736#gid=87243736",
  `databricks.connection` => "gdrive_demo_connection",
  format => "excel",
  schemaEvolutionMode => "none",
  -- optional parsing settings:
  headerRows => 1,
  inferSchema => true,
  dataAddress => "A1:E20"
);

-- Query the newly created table
SELECT * FROM gsheet_table_sql LIMIT 10

ORDERNUMBER,QUANTITYORDERED,PRICEEACH,ORDERLINENUMBER,SALES
10107,30,95.70,2,2871.00
10121,34,81.35,5,2765.90
10134,41,94.74,2,3884.34
10145,45,83.26,6,3746.70
10159,49,100.00,14,5205.27
10168,36,96.66,1,3479.76
10180,29,86.13,9,2497.77
10188,48,100.00,1,5512.32
10201,22,98.57,2,2168.54
10211,41,100.00,14,4708.44


---

## 3. Ingesta por Lotes de Múltiples Archivos CSV con el Mismo Esquema

Esta sección demuestra cómo:
1. Leer múltiples archivos CSV desde una carpeta de Google Drive
2. Combinarlos en una tabla Delta existente
3. Usar COPY INTO para carga incremental idempotente

Esto es útil cuando tienes múltiples archivos con la misma estructura que necesitan consolidarse en una sola tabla.

In [0]:
# Read all CSV files from a folder
csv_df = (spark.read
    .format("csv")
    .option("databricks.connection", "gdrive_demo_connection")
    .option("pathGlobFilter", "*.csv")  # Only read CSV files
    .option("recursiveFileLookup", True)  # Search subdirectories
    .option("inferSchema", True)  # Automatically infer column types
    .option("header", True)  # First row contains headers
    .load("https://drive.google.com/drive/u/5/folders/1zLrhqHY3UimMIlP15Yrjd0paAuDbHMuf")
)

# Create or replace the Delta table with CSV data
csv_df.write \
    .mode("overwrite") \
    .saveAsTable("gdrive_csv_data")

# Display the combined CSV data
display(csv_df)

TransactionID,Date,Product,Category,Quantity,UnitPrice,Total
TRX-1004,2023-04-06T06:17:00.000Z,HDMI Cable,Accessories,2,214,428
TRX-1007,2023-04-17T21:58:15.000Z,Mouse,Accessories,14,1279,17906
TRX-1022,2023-04-14T15:16:50.000Z,HDMI Cable,Accessories,16,155,2480
TRX-1030,2023-04-12T20:49:03.000Z,Laptop,Accessories,4,160,640
TRX-1035,2023-04-17T20:01:28.000Z,Monitor,Electronics,11,1298,14278
TRX-1044,2023-04-13T01:10:36.000Z,Monitor,Electronics,3,25,75
TRX-1047,2023-04-09T21:58:55.000Z,Keyboard,Accessories,6,111,666
TRX-1049,2023-04-17T07:10:33.000Z,Mouse,Electronics,2,984,1968
TRX-1006,2023-02-21T21:20:31.000Z,HDMI Cable,Accessories,3,1428,4284
TRX-1013,2023-02-05T15:57:15.000Z,HDMI Cable,Electronics,4,1158,4632


### Ingesta Incremental con COPY INTO

`COPY INTO` proporciona carga incremental idempotente - rastrea automáticamente qué archivos han sido procesados y solo carga archivos nuevos. Esto es perfecto para la ingesta continua de datos donde se agregan nuevos archivos CSV a una carpeta con el tiempo.

In [0]:
%sql
-- Create the table if it doesn't exist
CREATE TABLE IF NOT EXISTS gdrive_sample_csv_incremental;

-- Incrementally ingest new CSV files (only processes files not yet loaded)
COPY INTO gdrive_sample_csv_incremental
  FROM "https://drive.google.com/drive/u/5/folders/1zLrhqHY3UimMIlP15Yrjd0paAuDbHMuf"
  FILEFORMAT = CSV
  PATTERN = '*.csv'
  FORMAT_OPTIONS(
    'databricks.connection' = "gdrive_demo_connection",
    'header' = 'true',
    'inferSchema' = 'true')
  COPY_OPTIONS ('mergeSchema' = 'true');

-- Query the incrementally loaded data
SELECT * FROM gdrive_sample_csv_incremental LIMIT 10

---

## 4. Ingesta Automática e Incremental usando Lakeflow Spark Declarative Pipelines

Mientras que los ejemplos anteriores mostraron ingesta por lotes, **Lakeflow Spark Declarative Pipelines** te permite orquestar **ingesta automática e incremental** desde Google Drive. A medida que llegan nuevos archivos a un directorio de Google Drive (o se actualizan archivos), el pipeline los detecta e ingiere automáticamente.

### Beneficios Clave

* **Detección automática de archivos** - Los archivos nuevos o actualizados se descubren y procesan automáticamente
* **Procesamiento incremental** - Solo se procesan datos nuevos, no todo el conjunto de datos
* **Evolución de esquema** - Se adapta automáticamente a cambios de esquema en tus archivos
* **Orquestación** - Gestión de programación y dependencias integrada
* **Calidad de datos** - Expectativas y monitoreo incorporados en el pipeline

### Casos de Uso

* **Tablas de streaming** - Para ingesta continua de PDFs, CSVs u otros archivos a medida que llegan
* **Vistas materializadas** - Para archivos Excel específicos que se actualizan periódicamente (ej., reportes mensuales)

Los ejemplos a continuación muestran cómo definir tablas de pipeline usando sintaxis tanto en Python como en SQL.

### Lakeflow Spark Declarative Pipelines

Define tablas de streaming y vistas materializadas usando sintaxis SQL. Usa `CREATE OR REFRESH STREAMING TABLE` para ingesta continua y `CREATE OR REFRESH MATERIALIZED VIEW` para datos actualizados periódicamente.

NOTA: Los ejemplos de código a continuación deben copiarse a un pipeline SDP para ser desplegados. Consulta nuestra Documentación de Lakeflow Spark Declarative Pipelines para aprender más sobre cómo desplegar un pipeline SDP.

NOTA: Los usuarios deben configurar su canal de pipeline a "PREVIEW" (por defecto es "CURRENT") para usar esta característica en SDP. Pueden editarlo a través de la configuración del pipeline --> pestaña JSON


In [0]:
%sql
-- NOTE: You must copy the code into an SDP pipeline to deploy. View our documentation to learn more.

-- Incrementally ingest new PDF files as they arrive
CREATE OR REFRESH STREAMING TABLE gdrive_aircraft_maintenance_pdfs_streaming_sql
AS SELECT * FROM STREAM read_files(
  "https://drive.google.com/drive/u/5/folders/11gJMa1Nywn4gkiYrH4CS-aPZjvLBlDHj",
  format => "binaryFile",
  `databricks.connection` => "gdrive_demo_connection",
  pathGlobFilter => "*.pdf"
);

In [0]:
%sql
-- NOTE: You must copy the code into an SDP pipeline to deploy. View our documentation to learn more.

-- Incrementally ingest CSV files with automatic schema inference and evolution
CREATE OR REFRESH STREAMING TABLE gdrive_sample_csv_streaming_sql
AS SELECT * FROM STREAM read_files(
  "https://drive.google.com/drive/u/5/folders/1zLrhqHY3UimMIlP15Yrjd0paAuDbHMuf",
  format => "csv",
  `databricks.connection` => "gdrive_demo_connection",
  pathGlobFilter => "*.csv",
  header => "true"
);

In [0]:
%sql
-- NOTE: You must copy the code into an SDP pipeline to deploy. View our documentation to learn more.

-- Read a specific Google Sheet in a materialized view
-- This will automatically refresh when the Google Sheet is updated
CREATE OR REFRESH MATERIALIZED VIEW gdrive_sample_excel_materialized_sql
AS SELECT * FROM read_files(
  "https://docs.google.com/spreadsheets/d/1CQO4AzMthL-w0XWAcSkw4Eox6bo64rWVPCpzUflSsFA/edit?gid=87243736#gid=87243736",
  `databricks.connection` => "gdrive_demo_connection",
  format => "excel",
  headerRows => 1,
  `cloudFiles.schemaEvolutionMode` => "none"
);

---

## Próximos Pasos y Recursos

### Lo Que Has Aprendido

Has explorado exitosamente:
* ✅ Ingesta por lotes de PDFs, Google Sheets y archivos CSV desde Google Drive
* ✅ Análisis de documentos con IA usando `ai_parse_document()`
* ✅ Carga incremental con COPY INTO
* ✅ Ingesta automática y orquestada con Lakeflow Spark Declarative Pipelines

### Funciones de IA Adicionales para Explorar

Una vez que hayas analizado tus documentos, explora estas funciones de IA para análisis adicional:
* `ai_summarize()` - Generar resúmenes de contenido de texto
* `ai_extract()` - Extraer entidades específicas de documentos
* `ai_classify()` - Clasificar documentos en categorías
* `ai_analyze_sentiment()` - Analizar sentimiento en texto
* `ai_query()` - Consultar modelos fundamentales para análisis personalizado

### Recursos

* **Documentación**: [Guía del Conector de Google Drive](https://docs.google.com/document/d/e/2PACX-1vSV8pfG-Ib0A0QCsDJAUx3EoTlefMc6CVUdFrudgeVU5AcRPmvQggij2gbATuLN58F3b6yIvPS11D2M/pub)
* **Lakeflow Pipelines**: [Documentación de Spark Declarative Pipelines](https://www.databricks.com/product/data-engineering/spark-declarative-pipelines)
* **Funciones de IA**: [Referencia de Funciones de IA SQL](https://docs.databricks.com/aws/en/large-language-models/ai-functions)